# Phase 1 – Business Understanding

**CRISP-DM Step 1 of 5**

This notebook documents the business objectives, research questions, analytical goals, and success criteria for the European SME resource efficiency analysis project.

## 1. Background

Small and medium-sized enterprises (SMEs) account for the vast majority of businesses in Europe and are key drivers of economic activity and employment. Resource efficiency — how well firms convert inputs (energy, materials, water) into economic output — has become a strategic priority under the EU Green Deal, rising input costs, and growing regulatory pressure (e.g., the Corporate Sustainability Reporting Directive).

This project uses the **Flash Eurobarometer survey on SME resource efficiency**, a cross-sectional survey of **18,159 firms across 36 countries**, to identify what firm-level and contextual factors drive the adoption of resource efficiency practices among SMEs.

The survey is administered by Ipsos and archived by GESIS. It covers all EU-27 member states plus candidate countries, EFTA nations, and the USA.

**Analysis sample: EU-27 + UK (28 countries).** The UK is included because it only recently left the EU (Brexit 2020) and its regulatory environment, business culture, and market structure remain closely aligned with EU norms. Non-EU/UK countries (candidate states, EFTA, USA) are excluded.

The **nested structure** of the data (firms → countries → supranational macro-regions, cross-classified by sector) requires multilevel modelling to correctly partition variance and estimate effects.

**Supranational macro-regions** (Level 3 grouping):

| Region | Countries |
|--------|-----------|
| Northern | GB, DK, FI, IE, SE, EE, LV, LT |
| Western | AT, BE, DE, FR, LU, NL |
| Southern | CY, ES, GR, IT, MT, PT |
| CEE | BG, CZ, HR, HU, PL, RO, SI, SK |

## 2. Research Questions

### Primary question
> **Does the breadth of resource efficiency action adoption predict higher turnover growth in SMEs, after controlling for firm size, sector, age, and country?**

### Secondary questions
1. Do resource efficiency actions reduce self-reported production costs (Q3)?
2. Which specific RE action types (energy saving, recycling, renewable energy, etc.) are most strongly associated with positive outcomes?
3. How much of the variance in turnover change is explained by country vs. sector differences? (ICC)
4. Does the RE–turnover relationship differ by firm size (micro / small / medium)?
5. Is investment intensity in RE (share of annual revenue invested) a mediator of the RE–turnover relationship?
6. Are firms with green products and a climate neutrality strategy outperforming peers on turnover growth?

## 3. Analytical Framework

### 3.1 Primary Research Objective

> **Identify the firm-level and contextual factors that influence the *chance* and
> *intensity* of resource efficiency (RE) action adoption among European SMEs.**

- **Chance** — binary: does the firm adopt *any* RE action (`re_any_action`)?
- **Intensity** — count: how many of the 9 RE actions does the firm take (`re_action_count`)?
- **Choice** — per-action: which specific actions does each predictor drive?

Secondary analysis: do adopters show better business performance (turnover, costs)?

---

### 3.2 Dependent Variables (RE Adoption)

| Variable | Type | Description |
|----------|------|-------------|
| `re_action_count` | Ordinal count (0–9) | **Primary outcome — intensity** |
| `re_any_action` | Binary | Any adoption (adopter vs. non-adopter) |
| `re_water` | Binary | Saving water |
| `re_energy` | Binary | Saving energy |
| `re_renewables` | Binary | Using renewable energy |
| `re_materials` | Binary | Saving materials |
| `re_green_suppliers` | Binary | Switching to greener suppliers |
| `re_waste_min` | Binary | Minimising waste |
| `re_waste_sell` | Binary | Selling residues/waste |
| `re_recycling` | Binary | Recycling within company |
| `re_design` | Binary | Designing for reuse/repair |

**Action bundles** (sum scores capturing strategic coherence):

| Bundle | Actions | Max |
|--------|---------|-----|
| `bundle_energy` | Energy saving + Renewables | 2 |
| `bundle_circular` | Waste min/sell + Recycling + Design | 4 |
| `bundle_supply` | Green suppliers + Materials | 2 |
| `bundle_inputs` | Water + Energy + Materials | 3 |

---

### 3.3 Independent Variables (Firm Characteristics)

Predictors are selected via data-driven bivariate screening (Spearman ρ, point-biserial,
Kruskal-Wallis; Bonferroni-corrected) before entering the models.

| Variable | Source | Description | Role |
|----------|--------|-------------|------|
| `firm_size_ord` | SCR10 | 1=micro / 2=small / 3=medium | Structural |
| `firm_age_ord` | SCR12 | Reversed age category | Structural |
| `turnover_bracket_ord` | SCR14 | Baseline 2023 turnover (1–9 ord) | Financial capacity |
| `has_green_products` | Q9 | Offers green products/services | Strategic orientation |
| `has_climate_strategy` | Q14 | Has climate neutrality strategy | Strategic commitment |
| `uses_renewables` | Derived | Already uses renewable energy | Strategic orientation |
| `difficulty_count` | Derived | Count of perceived RE barriers (0–10) | Barrier capacity |
| `market_scope` | Derived | Domestic / EU / international | Market context |
| `re_investment_ord` | Q4 | RE investment intensity (1–6) | **Sensitivity only** (endogenous) |

**⚠ Circularity / endogeneity warnings:**
- `green_orientation` — composite sum includes RE action count itself → **excluded**
- `re_investment_ord` — may be consequence rather than cause of adoption → sensitivity only
- `turnover_change_ord` — old outcome variable; used only in secondary analysis

---

### 3.4 Modelling Strategy

Three levels of nesting; sector is cross-classified:

```
Level 3: EU Macro-region   (Northern / Western / Southern / CEE)
  └── Level 2: Country     (27 EU member states)
        └── Level 1: Firm  (n ≈ 14,000 EU SMEs)

Cross-classified: Sector   (4 NACE groups)
```

**Model sequence:**

| Model | Outcome | Predictors | Random structure | Question |
|-------|---------|-----------|-----------------|----------|
| M0 | `re_action_count` | — | RI: country / macro / sector | How much variance is contextual? |
| M1 | `re_action_count` | Screened firm chars | RI: country | Which factors drive intensity? |
| M2 | `re_action_count` | + macro-region FEs | RI: country | Do EU blocs differ after controlling firm chars? |
| M3 | `re_action_count` | Firm chars | RI: country × sector | Country vs. sector-specific adoption norms? |
| M4 | `re_action_count` | Firm chars | RI + RS: country | Does the effect of the top predictor vary by country? |
| M5 | `re_any_action` | Firm chars | RI: country (GLMM) | What distinguishes adopters from non-adopters? |
| Per-action ×9 | each `re_*` | Firm chars | RI: country (GLMM) | Which predictors drive which specific actions? |
| Bundles ×4 | each bundle | Firm chars | RI: country | Do factors differ by action strategy type? |

**Predictor selection steps (in NB04):**
1. Spearman ρ / point-biserial / Kruskal-Wallis for all candidates vs. `re_action_count`
2. Apply Bonferroni correction (α/n_tests); retain p < corrected threshold
3. VIF check on retained set; drop VIF > 5
4. `re_investment_ord` excluded from primary models regardless of p-value

**Key outputs:**
- Forest plot: M1 predictor effects on adoption breadth
- Coefficient heatmap: 9 actions × n predictors (per-action GLMMs)
- Caterpillar plot: country-level random intercepts (which countries systematically over/under-adopt?)

## 4. Success Criteria

| Criterion | Target |
|-----------|---------|
| ICC justifies multilevel modelling | ICC > 0.05 at country or sector level |
| RE action count predicts turnover change | Significant positive coefficient (p < 0.05) |
| Models converge without warnings | — |
| Residuals approximately normal | Visual Q-Q inspection |
| AIC improvement M1 → M3 | ΔAIC > 10 |
| Deliverable: processed dataset | Saved to `data/processed/sme_model_ready.parquet` |
| Deliverable: all 5 notebooks run end-to-end | — |

## 5. Survey Design Notes

- **Sampling**: Stratified by country, firm size (employees), and NACE sector
- **Mode**: CATI / online (mixed mode)
- **Reference period**: Survey conducted in 2024; turnover/cost questions refer to the past 2–3 years
- **Weights**: `w1` (all companies), `w1_sme` (SMEs only) — apply for descriptive statistics; for HLM we use unweighted data and control for design variables
- **Skip patterns**: Q3, Q4, Q5–Q7 are only asked if the firm took at least one RE action (Q1.1–Q1.9 ≥ 1). Structural missingness is coded as `9 = Inapplicable` — not random missing.
- **DK/NA codes**: 997, 998, 999, 999997, 999998 must be recoded to `NaN` before analysis.

## 6. Project Plan (CRISP-DM)

```
Phase 1: Business Understanding  ← (this notebook)
Phase 2: Data Understanding      → 02_data_understanding.ipynb
Phase 3: Data Preparation        → 03_data_preparation.ipynb
Phase 4: Modelling               → 04_modeling.ipynb
Phase 5: Evaluation              → 05_evaluation.ipynb
```